# Calculating RMST Pseudo-Observations

**This notebook calculates RMST pseudo-observations at 1 and 2 years for advanced melanoma patients receiving combined immunotherapy or targeted therapy.**

In [1]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time
from joblib import Parallel, delayed

from utils.pseudo_obs import pseudo_observations_rmst

In [2]:
df = pd.read_csv('../outputs/ioio_tki_features_df.csv')

In [3]:
df = df.set_index('PatientID')

In [4]:
df.shape

(1339, 173)

In [5]:
treatment_df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [6]:
treatment_df.shape

(2771, 3)

In [7]:
df = pd.merge(df, treatment_df, on = 'PatientID', how = 'left')

In [8]:
df.shape

(1339, 176)

In [9]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [10]:
df['treatment_year'] = df['StartDate'].dt.year

In [11]:
df = df.query('treatment_year <= 2021')

In [12]:
df.shape

(1069, 177)

In [13]:
pseudo_rmst = pseudo_observations_rmst(
    df['duration'].values,
    df['event'].values,
    tau = 365
)

df['rmst_pseudo_1y'] = pseudo_rmst

In [14]:
pseudo_rmst = pseudo_observations_rmst(
    df['duration'].values,
    df['event'].values,
    tau = 730
)

df['rmst_pseudo_2y'] = pseudo_rmst

In [15]:
df = df.reset_index()

In [16]:
df = df[['PatientID', 'rmst_pseudo_1y', 'rmst_pseudo_2y']]

In [17]:
df.head(5)

,PatientID,rmst_pseudo_1y,rmst_pseudo_2y
0,F744F618949B5,367.956504,744.340936
1,F4AAE7EB8AE49,357.021846,631.659309
2,F702FE1F825B7,367.956504,744.340936
3,F51E165560B72,367.956504,744.340936
4,F13FF1FBBFA17,367.956504,744.340936


In [18]:
df.to_csv('../outputs/pseudo_obs_rmst.csv', index = False)